General Flow of Code per PDF build
- Collect questions to be included in PDF
- Generate dictionary for question
- Save dictionary to pdf_database
- Create latex file by including question per pdf_database item
- Run latex file to create pdf
- Save pdf to location

## Testing single questions

In [2]:
response_type = 'Multiple-Choice'
version = 'A'

from learning_objective_code.real_complex_numbers import divide_complex_numbers

test_dict, test_df = divide_complex_numbers.divide_complex_numbers_function(response_type, 0)

In [3]:
test_dict

{'code_name': 'divide_complex_numbers',
 'Response Type': 'Multiple-Choice',
 'Display Stem Type': 'String',
 'Display Stem': 'Simplify the expression below into the form $a+bi$. Then, choose the letter that matches your expression.',
 'Display Problem Type': 'Math Mode',
 'Display Problem': '\\frac{-45  + 44 i}{6  - 7 i}',
 'Display Options Type': 'Math Mode',
 'Solution': '-6.80  - 0.60 i',
 'Answer Letter': 'C',
 'General Comment': 'Multiply the numerator and denominator by the *conjugate* of the denominator, then simplify. For example, if we have $2+3i$, the conjugate is $2-3i$.'}

In [4]:
test_df

,code_name,name,short_description,values_for_interval_generation,value,feedback,solution,choice_presentation,letter
0,divide_complex_numbers,solution,Expected solution,"[-6.8, -0.6]",-6.80 - 0.60 i,"$-6.80 - 0.60 i$, which is the correct option.",1,"a \in [-7.4, -6.1] \text{ and } b \in [-1.5, 0]",C
1,divide_complex_numbers,distractor_1,Divide like terms,"[-7.5, -6.285714285714286]",-7.50 - 6.29 i,"$-7.50 - 6.29 i$, which corresponds to just ...",0,"a \in [-8.35, -6.85] \text{ and } b \in [-6.5,...",A
2,divide_complex_numbers,distractor_2,Multiply by non-conjugate and treat like conju...,"[0.4470588235294118, 6.811764705882353]",0.45 + 6.81 i,"$0.45 + 6.81 i$, which corresponds to forget...",0,"a \in [-0.05, 1.05] \text{ and } b \in [5, 8.5]",D
3,divide_complex_numbers,distractor_3,"Multiply by conjugate, only divide first term","[-6.8, -51.0]",-6.80 - 51.00 i,"$-6.80 - 51.00 i$, which corresponds to forg...",0,"a \in [-7.4, -6.1] \text{ and } b \in [-52.5, ...",E
4,divide_complex_numbers,distractor_4,"Multiply by conjugate, only divide second term","[-578.0, -0.6]",-578.00 - 0.60 i,"$-578.00 - 0.60 i$, which corresponds to for...",0,"a \in [-578.05, -577.8] \text{ and } b \in [-1...",B


## Running Exam

In [5]:
import os
import pandas as pd

base_dir = os.getcwd()
question_directory_path = os.path.join(base_dir, 'learning_objective_code', 'question_directory.xlsx')
question_directory_df = pd.read_excel(question_directory_path, index_col=0)

In [6]:
exam_name = 'Test Exam Generation'
file_name = 'test_file_name'
footnote_left = 'UUID'
footnote_right = "Today's Date"
test_modules = [1, 2, 3, 4]
version = 'A'
interval_options = 0

In [7]:
def create_modules_mask(list_of_modules):
    final_mask = question_directory_df['module_number'] == list_of_modules[0]
    list_of_modules.pop(0)
    while len(list_of_modules) > 0:
        final_mask = final_mask | (question_directory_df['module_number'] == list_of_modules[0])
        list_of_modules.pop(0)
    return final_mask

In [8]:
questions_to_create_df = question_directory_df[create_modules_mask(test_modules)].copy()
questions_to_create_df.reset_index(drop=True, inplace=True)
questions_to_create_df.reset_index(drop=False, inplace=True, names=['question_number'])

questions_to_create_df['response_type'] = 'Multiple-Choice'
questions_to_create_df['version'] = version
questions_to_create_df['interval_options'] = interval_options
save_questions_df_file_path = os.path.join(base_dir, 'temp_files', 'build_exams', 'questions_to_create_df.xlsx')
questions_to_create_df.to_excel(save_questions_df_file_path)

In [9]:
from utils import question_script_generation

exam_path = os.path.join(base_dir, file_name)
question_script_generation.generate_question_running_script(exam_path, questions_to_create_df)

In [10]:
import subprocess

subprocess.run(['python', f'{exam_path}.py'])

CompletedProcess(args=['python', 'c:\\Users\\dcham\\Documents\\GitHub\\CollegeAlgebraLSLTAnalysis\\autodig\\test_file_name.py'], returncode=0)

In [11]:
question_script_generation.move_files(exam_path)

In [12]:
from utils import start_end_latex_files, add_questions_to_latex, build_latex_files

start_end_latex_files.start_all_latex_files(file_name, exam_name, footnote_left, footnote_right, version, base_dir)
add_questions_to_latex.print_all_questions_to_latex_files(file_name, base_dir, interval_options)
start_end_latex_files.end_all_latex_files(file_name, version, base_dir)
build_latex_files.build_latex_files_function(base_dir, file_name, version)